# XGBoost — second version (ratio 2:1, following the `tuning/v3` protocol)

## Tuning configuration
- **Dataset:** **2:1** pseudo-absences:presences ratio (equivalent to "1 presence for
  every 2 pseudo-absences") — the ratio most representative of the real prevalence
  (~2.5%), rebuilt here with the SAME logic (same base table `pixel_year_full.csv`,
  same 3 km exclusion buffer, same `random_state=42` seed) so it is comparable
  figure by figure with the `2:1` rows for LR and RF already reported in `tuning/v3`.
- **Hyperparameter search:** `RandomizedSearchCV` with **150 configurations**
  randomly sampled (out of 300 possible combinations), `GroupKFold(10)` over 0.25°
  spatial blocks, `scoring='average_precision'`, **restricted to `year<=2019`**
  (same temporal-leakage correction as v2/v3).
- **`scale_pos_weight` = n_negatives/n_positives (≈2.0)**: with a 2:1 ratio the
  dataset is NOT balanced (twice as many pseudo-absences as presences), so the
  positive class is weighted to compensate — this is the exact equivalent of the
  `class_weight='balanced'` used by LR and RF at 2:1, so the comparison is fair.

### Note on the requested grid
The original grid was specified in Random Forest terms (`n_estimators`, `max_depth`,
`max_features`, `min_samples_leaf`). XGBoost has neither `max_features` nor
`min_samples_leaf` — its native boosting equivalents are used, preserving the
same values where possible:

| Requested parameter (RF) | Requested values | XGBoost equivalent | Values used |
|---|---|---|---|
| `n_estimators` | 100, 150, 200, 250, 350 | `n_estimators` (same name and values) | 100, 150, 200, 250, 350 |
| `max_depth` | None, 10, 20, 30 | `max_depth` (0 = no limit in XGBoost) | 0, 10, 20, 30 |
| `max_features` | 'sqrt', 0.5, 4 | `colsample_bytree` (fraction of columns per tree) | √9/9≈0.333, 0.5, 4/9≈0.444 |
| `min_samples_leaf` | 4, 6, 10, 20, 30 | `min_child_weight` (minimum weight/samples per leaf) | 4, 6, 10, 20, 30 |

$5 \times 4 \times 3 \times 5 = 300$ possible combinations; 150 are sampled at random
(150 × 10 folds = 1,500 fits), a scale comparable to the 720 RF fits in v1/v3.

## Methodological note on PR-AUC across ratios
The PR-AUC of this 2:1 dataset **is not directly comparable** with that of the 1:1
XGBoost run, because the PR-AUC baseline is the prevalence of the positive class:
0.33 at 2:1 versus 0.50 at 1:1. PR-AUC at 2:1 will look lower by construction, NOT
because of worse performance. The valid comparison *across ratios* is done with
AUC-ROC (invariant to prevalence). The comparison *within* this ratio (XGBoost vs.
LR vs. RF, all at 2:1) is valid on PR-AUC because they share the same prevalence.

## What this notebook does
1. Rebuilds the 2:1 ratio dataset from `pixel_year_full.csv` (identical to `tuning/v3`).
2. Tunes XGBoost with `RandomizedSearchCV` (150 configurations, `year<=2019`).
3. Evaluates the tuned model with the full protocol (spatial block CV, 10 folds +
   temporal hold-out `>=2020`, with confusion matrices) — same as LR and RF.
4. Saves the results in this folder (`model/xgboost/`): `xgboost_metrics_2to1.csv` and
   `xgboost_vs_lr_rf_comparison_2to1.csv` (direct comparison against LR and RF at ratio 2:1).

In [8]:
# === train_xgboost.ipynb — Setup: 2:1 ratio dataset (same protocol as tuning/v3) ===
# Rebuilds EXACTLY the same 2:1 dataset that tuning/v3/tune_v3.ipynb generated for
# Random Forest and Logistic Regression (same base table, same 3 km exclusion
# buffer, same random_state=42 seed), so XGBoost is directly comparable.
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                              confusion_matrix)
from sklearn.base import clone

pixel_year = pd.read_csv('../../data/model_dataset/pixel_year_full.csv')
pred_cols = ['dist_roads','dist_parks','dist_coca','dist_mosaic',
             'temp_C','vpd_kPa','ndvi','wind_ms','oni']
BLOCK = 0.25
EXCL_BUFFER = 3000            # 3 km — identical to the one used in tuning/v3
buffer_deg = EXCL_BUFFER / 111000
RATIO = 2                     # 2 pseudo-absences per presence (project label "2:1")
SEED = 42

presences = pixel_year[pixel_year['burned'] == 1].copy()
absences_all = pixel_year[pixel_year['burned'] == 0].copy()

kept_absences = []
for y in sorted(pixel_year['year'].unique()):
    pres_y = presences[presences['year'] == y][['lon','lat']].values
    abs_y  = absences_all[absences_all['year'] == y]
    if len(pres_y) == 0:
        kept_absences.append(abs_y); continue
    tree = cKDTree(pres_y)
    dists, _ = tree.query(abs_y[['lon','lat']].values, k=1)
    kept_absences.append(abs_y[dists > buffer_deg])
absences_far = pd.concat(kept_absences, ignore_index=True)

n_abs = min(len(absences_far), int(round(RATIO * len(presences))))
absences_sample = absences_far.sample(n=n_abs, random_state=SEED)
model_df = pd.concat([presences, absences_sample], ignore_index=True) \
             .sample(frac=1, random_state=SEED).reset_index(drop=True)
model_df['block'] = (model_df['lon']//BLOCK).astype(int).astype(str) + '_' + \
                     (model_df['lat']//BLOCK).astype(int).astype(str)

n_pres = int(model_df['burned'].sum())
n_neg  = model_df.shape[0] - n_pres
print("Dataset ratio 2:1 ->", model_df.shape,
      f"({n_pres} presences, {n_neg} pseudo-absences)")

Dataset ratio 2:1 -> (6231, 14) (2077 presencias, 4154 pseudo-ausencias)


In [9]:
# === RandomizedSearchCV — 150 random configurations, tuning ONLY with year<=2019 ===
tune_df = model_df[model_df['year'] <= 2019].copy()
X_tune = tune_df[pred_cols].values
y_tune = tune_df['burned'].astype(int).values
groups_tune = tune_df['block'].values
cv = GroupKFold(n_splits=10)

# 2:1 dataset is imbalanced -> weight the positive class (equivalent to class_weight='balanced')
SCALE_POS_WEIGHT = n_neg / n_pres
print(f"scale_pos_weight = n_neg/n_pos = {n_neg}/{n_pres} = {SCALE_POS_WEIGHT:.3f}")

param_distributions_xgb = {
    'n_estimators':     [100, 150, 200, 250, 350],
    'max_depth':        [0, 10, 20, 30],                 # 0 = no limit (equivalent to None in RF)
    'colsample_bytree': [np.sqrt(len(pred_cols)) / len(pred_cols), 0.5, 4 / len(pred_cols)],
    'min_child_weight': [4, 6, 10, 20, 30],
}
n_combinations = (len(param_distributions_xgb['n_estimators']) *
                   len(param_distributions_xgb['max_depth']) *
                   len(param_distributions_xgb['colsample_bytree']) *
                   len(param_distributions_xgb['min_child_weight']))

xgb_base = XGBClassifier(
    objective='binary:logistic',
    eval_metric='aucpr',
    scale_pos_weight=SCALE_POS_WEIGHT,     # imbalanced 2:1 dataset -> class correction
    tree_method='hist',
    random_state=42,
    n_jobs=-1,
)

xgb_search = RandomizedSearchCV(
    xgb_base,
    param_distributions=param_distributions_xgb,
    n_iter=150,
    scoring='average_precision',
    cv=cv,
    random_state=42,
    n_jobs=-1,
    refit=True,
)
xgb_search.fit(X_tune, y_tune, groups=groups_tune)

print(f"Grid: {n_combinations} possible combinations -> 150 sampled x 10 folds = "
      f"{150 * 10} fits")
print("Best XGBoost hyperparameters:", xgb_search.best_params_)
print(f"Best PR-AUC (tuning CV, year<=2019): {xgb_search.best_score_:.3f}")

scale_pos_weight = n_neg/n_pos = 4154/2077 = 2.000
Grilla: 300 combinaciones posibles -> 150 muestreadas x 10 folds = 1500 fits
Mejores hiperparámetros XGBoost: {'n_estimators': 100, 'min_child_weight': 30, 'max_depth': 30, 'colsample_bytree': 0.4444444444444444}
Mejor PR-AUC (CV tuning, year<=2019): 0.777


In [10]:
def evaluate_model(name, estimator, model_df, pred_cols, n_splits=10, block=BLOCK):
    """Validation protocol IDENTICAL to tuning/v3 (spatial block CV with 10 folds
    over the full dataset + temporal hold-out train<=2019/test>=2020), with
    confusion matrices — so XGBoost is directly comparable with LR and RF."""
    X = model_df[pred_cols].values
    y = model_df['burned'].astype(int).values
    blocks = (model_df['lon']//block).astype(int).astype(str) + '_' + \
             (model_df['lat']//block).astype(int).astype(str)
    groups = blocks.values

    gkf = GroupKFold(n_splits=n_splits)
    rows = []
    cm_spatial = np.zeros((2, 2), dtype=int)
    for tr, te in gkf.split(X, y, groups):
        m = clone(estimator).fit(X[tr], y[tr])
        prob = m.predict_proba(X[te])[:, 1]
        pred = m.predict(X[te])
        auc = roc_auc_score(y[te], prob)
        prauc = average_precision_score(y[te], prob)
        f1 = f1_score(y[te], pred)
        rows.append((auc, prauc, f1))
        cm_spatial += confusion_matrix(y[te], pred, labels=[0, 1])
    r = np.array(rows)
    print(f"  [{name}] SPATIAL  AUC={r[:,0].mean():.3f}±{r[:,0].std():.3f}  "
          f"PR-AUC={r[:,1].mean():.3f}±{r[:,1].std():.3f}  F1={r[:,2].mean():.3f}±{r[:,2].std():.3f}")

    tr = (model_df['year'] <= 2019).values
    te = (model_df['year'] >= 2020).values
    m = clone(estimator).fit(X[tr], y[tr])
    prob = m.predict_proba(X[te])[:, 1]
    pred = m.predict(X[te])
    cm_temporal = confusion_matrix(y[te], pred, labels=[0, 1])
    auc_t = roc_auc_score(y[te], prob)
    prauc_t = average_precision_score(y[te], prob)
    f1_t = f1_score(y[te], pred)
    print(f"  [{name}] TEMPORAL AUC={auc_t:.3f}  PR-AUC={prauc_t:.3f}  F1={f1_t:.3f}")

    return {
        'spatial': r.mean(axis=0), 'spatial_std': r.std(axis=0), 'spatial_cm': cm_spatial,
        'temporal': (auc_t, prauc_t, f1_t), 'temporal_cm': cm_temporal,
    }


def cm_to_dict(cm, prefix):
    tn, fp, fn, tp = cm.ravel()
    return {f'{prefix}_tn': int(tn), f'{prefix}_fp': int(fp),
            f'{prefix}_fn': int(fn), f'{prefix}_tp': int(tp)}


res_xgb = evaluate_model("XGBoost (tuned, ratio 2:1)", xgb_search.best_estimator_, model_df, pred_cols)

  [XGBoost (tuned, ratio 2:1)] SPATIAL  AUC=0.862±0.039  PR-AUC=0.745±0.073  F1=0.694±0.060
  [XGBoost (tuned, ratio 2:1)] TEMPORAL AUC=0.794  PR-AUC=0.540  F1=0.555


In [ ]:
# === Save results — same format as tuning_v3_final_metrics.csv ===
row = {
    'ratio': '2:1',
    'model': 'XGBoost',
    'best_params': str(xgb_search.best_params_),
    'n_rows': model_df.shape[0],
    'n_presences': n_pres,
    'auc_spatial_mean': res_xgb['spatial'][0], 'auc_spatial_std': res_xgb['spatial_std'][0],
    'prauc_spatial_mean': res_xgb['spatial'][1], 'prauc_spatial_std': res_xgb['spatial_std'][1],
    'f1_spatial_mean': res_xgb['spatial'][2], 'f1_spatial_std': res_xgb['spatial_std'][2],
    'auc_temporal': res_xgb['temporal'][0], 'prauc_temporal': res_xgb['temporal'][1],
    'f1_temporal': res_xgb['temporal'][2],
}
row.update(cm_to_dict(res_xgb['spatial_cm'], 'cm_spatial'))
row.update(cm_to_dict(res_xgb['temporal_cm'], 'cm_temporal'))

xgb_metrics_df = pd.DataFrame([row])
xgb_metrics_df.to_csv('xgboost_metrics_2to1.csv', index=False)
print("Results saved to: model/xgboost/xgboost_metrics_2to1.csv")
xgb_metrics_df[['ratio','model','n_rows','n_presences','auc_spatial_mean','prauc_spatial_mean',
                'f1_spatial_mean','auc_temporal','prauc_temporal','f1_temporal']]

Resultados guardados en: model/xgboost/xgboost_metrics_2to1.csv


,ratio,model,n_rows,n_presences,auc_spatial_mean,prauc_spatial_mean,f1_spatial_mean,auc_temporal,prauc_temporal,f1_temporal
0,2:1,XGBoost,6231,2077,0.861716,0.74494,0.694382,0.793558,0.540112,0.554572


In [ ]:
# === Direct comparison against LR and RF (same 2:1 ratio, same validation protocol) ===
v3_sensitivity = pd.read_csv('../../tuning/v3/tuning_v3_sensitivity_metrics.csv')
cols = ['ratio','model','n_rows','n_presences','auc_spatial_mean','prauc_spatial_mean',
        'f1_spatial_mean','auc_temporal','prauc_temporal','f1_temporal']
lr_rf_21 = v3_sensitivity[v3_sensitivity['ratio'] == '2:1'][cols]

comparison_df = pd.concat([lr_rf_21, xgb_metrics_df[cols]], ignore_index=True)
comparison_df.to_csv('xgboost_vs_lr_rf_comparison_2to1.csv', index=False)
print("Comparison saved to: model/xgboost/xgboost_vs_lr_rf_comparison_2to1.csv\n")
comparison_df

Comparación guardada en: model/xgboost/xgboost_vs_lr_rf_comparison_2to1.csv



,ratio,model,n_rows,n_presences,auc_spatial_mean,prauc_spatial_mean,f1_spatial_mean,auc_temporal,prauc_temporal,f1_temporal
0,2:1,Logistic Regression,6231,2077,0.832055,0.702716,0.613300,0.808193,0.616809,0.557166
1,2:1,Random Forest,6231,2077,0.876874,0.779414,0.680450,0.819263,0.568723,0.560000
2,2:1,XGBoost,6231,2077,0.861716,0.744940,0.694382,0.793558,0.540112,0.554572


## Conclusions

**Best hyperparameters (`RandomizedSearchCV`, 150/300 combinations, tuning `year<=2019`): XGBoost: {'n_estimators': 100, 'min_child_weight': 30, 'max_depth': 30, 'colsample_bytree': 0.4444444444444444}
Best PR-AUC (tuning CV, year<=2019): 0.777

### Results (ratio 2:1, dataset identical to LR/RF in `tuning/v3`)

| Model | Spatial AUC | Spatial PR-AUC | Spatial F1 | Temporal AUC | Temporal PR-AUC | Temporal F1 |
|---|---|---|---|---|---|---|
| Logistic Regression | 0.83 | 0.70 | 0.61 | 0.81 | **0.62** | 0.56 |
| Random Forest | **0.88** | **0.78** | 0.68 | **0.82** | 0.57 | 0.56 |
| XGBoost | 0.86 | 0.74 | **0.69** | 0.79 | 0.54 | 0.55 |

(full table in [`xgboost_vs_lr_rf_comparison_2to1.csv`](xgboost_vs_lr_rf_comparison_2to1.csv))

### How to read these numbers
- **PR-AUC at 2:1 vs 1:1:** PR-AUC here is measured against a prevalence baseline of
  **0.33** (not 0.50 as at 1:1). That is why the 2:1 values look lower than in the
  1:1 notebook — it is a prevalence effect, NOT worse performance. To compare
  *across ratios*, AUC-ROC is used (invariant to prevalence); to compare *within*
  2:1 (XGBoost vs LR vs RF) PR-AUC is valid because they share prevalence.
- **Project priority = temporal PR-AUC** (the `>=2020` hold-out is the validation
  that most closely resembles predicting an unobserved future year, like the 2026 map).

### Does XGBoost improve the prediction?

- **Spatial validation:** the ranking is **Random Forest (PR-AUC 0.78) > XGBoost (0.74) >
  Logistic Regression (0.70)**. Both tree-based models outperform the linear baseline,
  confirming that non-linear relationships add value in predicting *where* fires occur.
  However, XGBoost **does not outperform Random Forest**: it is 0.04 below in spatial
  PR-AUC, a difference comparable to the standard deviation between folds.
- **Temporal validation (train ≤2019 / test ≥2020):** XGBoost gets the **lowest
  temporal PR-AUC of the three** (0.54, versus 0.57 for RF and 0.62 for LR). Unlike
  the 1:1 ratio — where XGBoost slightly outperformed RF (0.717 vs 0.704) — at 2:1
  that advantage **disappears**. This shows that XGBoost's edge at 1:1 was **fragile
  and dependent on the sampling ratio**, not a robust algorithmic improvement.
- **Logistic Regression achieves the best temporal PR-AUC** (0.62), ahead of both
  non-linear models. This reinforces the project's finding: in the temporal dimension
  the interannual signal is weak, and the complex models fit noise that does not
  generalize to new years, while the more rigid linear baseline generalizes better.

### Conclusion for the 2026 susceptibility map

**Random Forest is the final selected model.** It wins on spatial discrimination
(PR-AUC 0.78, AUC 0.88) — the direct objective of the susceptibility map —, is
competitive on temporal validation, and its performance is **robust to the choice of
sampling ratio**, unlike XGBoost, whose slight edge at 1:1 did not hold at 2:1.
It is also simpler to interpret and less prone to overfitting with little data
(~6,231 rows). XGBoost remains an equivalent non-linear alternative but without a
demonstrable advantage; Logistic Regression remains the transparent baseline.

This three-model comparison over an identical dataset and protocol closes the model
evaluation phase. It confirms two central project conclusions: (1) non-linear models
improve **spatial** prediction over the baseline, justifying the ML approach; and
(2) the **sampling ratio matters more than the choice of algorithm**, to the point of
reversing the ranking between RF and XGBoost. The next step is interpreting the
winning model via **SHAP + permutation importance**, followed by susceptibility
classification (Jenks) and generation of the 2026 map.